# Comprehensions: List and Dict

**Session 06 (2026-10-24) · Priority 2 · Notebook 5 of 6**

A comprehension builds a new list or dictionary from an existing collection in **one readable line**, instead of a 3 to 4 line loop.

What you'll learn:

1. What a comprehension is, and what it does behind the scenes
2. List comprehensions
3. Filtering with `if`
4. Transforming with `if` / `else`
5. Dict comprehensions
6. Bonus: set comprehensions and generator expressions
7. Nested comprehensions
8. When **not** to use a comprehension
9. Practice

---
## 1. What is a comprehension?

**What it is:** a short way to build a new list (or dict, or set) by looping over an iterable, written as a single expression.

**Behind the scenes:** a list comprehension does exactly what a `for` loop that appends to an empty list does. Python runs the loop for you and hands back the finished list. It's usually a little faster than the hand-written loop too, because the appending happens inside Python itself.

In [ ]:
# The loop way
squares = []
for n in range(1, 6):
    squares.append(n * n)
print(squares)

# The comprehension way: the same result
squares = [n * n for n in range(1, 6)]
print(squares)

Read it as: **"`n * n` for each `n` in `range(1, 6)`"**. The pieces of the loop map straight across:

```
squares = []
for n in range(1, 6):        →    squares = [ n * n    for n in range(1, 6) ]
    squares.append(n * n)                    └ what ┘  └─── the loop ──────┘
                                              to keep
```

---
## 2. List comprehensions

**Syntax:** `[expression for item in iterable]`
**Returns:** a **new** list. The original iterable is not changed.

In [ ]:
names = ["asha", "ravi", "monal"]

upper_names = [name.upper() for name in names]
lengths = [len(name) for name in names]

print(upper_names)
print(lengths)
print(names)                    # the original is unchanged

A very common use is converting types, like the comma-separated numbers from the strings lesson:

In [ ]:
values = "1,2,3,4,5"
numbers = [int(v) for v in values.split(",")]
print(numbers, sum(numbers))

**Behind the scenes: the loop variable stays inside.** After a normal `for` loop, the loop variable still exists. A comprehension's variable exists only inside the comprehension:

In [ ]:
for i in range(3):
    pass
print("after a for loop, i =", i)      # i still exists

squares = [j * j for j in range(3)]
try:
    print(j)
except NameError as e:
    print("NameError:", e)             # j only existed inside the comprehension

---
## 3. Filtering with `if`

**Syntax:** `[expression for item in iterable if condition]`

Only items where `condition` is `True` make it into the new list.

In [ ]:
numbers = list(range(1, 11))

evens = [n for n in numbers if n % 2 == 0]
big_squares = [n * n for n in numbers if n > 5]

print(evens)
print(big_squares)

The loop equivalent of `evens`, for comparison:

```python
evens = []
for n in numbers:
    if n % 2 == 0:
        evens.append(n)
```

A real-world example: clean up a list of emails, then keep only the Gmail ones.

In [ ]:
emails = ["asha@gmail.com", "ravi@yahoo.com", "monal@gmail.com", "  ALEX@Gmail.com "]

clean = [e.strip().lower() for e in emails]
gmail_users = [e for e in clean if e.endswith("@gmail.com")]

print(clean)
print(gmail_users)

---
## 4. Transforming with `if` / `else`

An `if` at the **end** filters: it drops items. To **choose between two values** for every item, put a conditional expression (from the control flow lesson) at the **front**:

`[value_if_true if condition else value_if_false for item in iterable]`

In [ ]:
marks = [92, 45, 78, 30, 66]

results = ["pass" if m >= 50 else "fail" for m in marks]   # one result per mark
passed = [m for m in marks if m >= 50]                     # only some of the marks

print(results)
print(passed)

| Where the `if` goes | Effect | Length of the result |
|---|---|---|
| At the end: `[x for x in xs if cond]` | **filter** | can be shorter than the input |
| At the front: `[a if cond else b for x in xs]` | **transform** | always the same as the input |

Putting `else` at the end, as in `[x for x in xs if cond else y]`, is a `SyntaxError`.

---
## 5. Dict comprehensions

**Syntax:** `{key_expression: value_expression for item in iterable}`, optionally with `if condition` at the end.
**Returns:** a new dictionary.

In [ ]:
names = ["asha", "ravi", "monal"]

name_lengths = {name: len(name) for name in names}
print(name_lengths)

Build a dict from two lists with `zip()`, which pairs them up item by item (covered in notebook 06):

In [ ]:
students = ["asha", "ravi", "monal"]
marks = [92, 45, 78]

marks_by_student = {s: m for s, m in zip(students, marks)}
print(marks_by_student)

Loop over an existing dict with `.items()` to filter it, transform its values, or swap keys and values:

In [ ]:
prices = {"apple": 120, "banana": 40, "mango": 150, "grapes": 90}

expensive = {item: price for item, price in prices.items() if price > 100}
discounted = {item: price * 0.9 for item, price in prices.items()}
swapped = {price: item for item, price in prices.items()}

print(expensive)
print(discounted)
print(swapped)

The dictionary rule still applies: if two items produce the **same key**, the last one wins.

In [ ]:
words = ["apple", "avocado", "banana"]
by_first_letter = {w[0]: w for w in words}
print(by_first_letter)       # 'a' -> 'avocado': it overwrote 'apple'

---
## 6. Bonus: set comprehensions and generator expressions

| Syntax | Builds |
|---|---|
| `[x for x in xs]` | a list |
| `{x for x in xs}` | a set (unique values only) |
| `{k: v for ...}` | a dict |
| `(x for x in xs)` | a **generator** (not a tuple!) |

**A generator expression** doesn't build a collection at all. **Behind the scenes**, it produces one value at a time, only when asked, so it uses almost no memory. That's ideal when you pass the values straight into a function like `sum()`, `max()` or `any()`. It can be used **only once**.

In [ ]:
words = ["Apple", "apple", "APPLE", "banana"]
unique_lower = {w.lower() for w in words}         # set comprehension
print(unique_lower)

total = sum(n * n for n in range(1, 1_000_001))   # no million-item list is built
print(total)

---
## 7. Nested comprehensions

Several `for` clauses read **left to right**, in the same order as nested loops:

In [ ]:
# Loop version
pairs = []
for color in ["red", "blue"]:
    for size in ["S", "M"]:
        pairs.append((color, size))
print(pairs)

# Comprehension version: the for clauses are in the same order
pairs = [(color, size) for color in ["red", "blue"] for size in ["S", "M"]]
print(pairs)

A common use is **flattening** a list of lists:

In [ ]:
matrix = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
flat = [n for row in matrix for n in row]
print(flat)

A comprehension **inside** a comprehension builds a nested list:

In [ ]:
table = [[row * col for col in range(1, 4)] for row in range(1, 4)]
print(table)        # a 3 x 3 multiplication table

---
## 8. When not to use a comprehension

Comprehensions are for **building a new collection**. Use a normal loop when:

- the logic needs several steps, or more than one simple condition, so a one-liner would be hard to read
- you want a **side effect** (printing, writing to a file) and don't need the resulting list
- you need `try` / `except` for each item (a comprehension can't contain statements)

In [ ]:
# Don't: this builds a useless list of None values just to print
result = [print(n) for n in range(3)]
print(result)

# Do: a plain loop
for n in range(3):
    print(n)

**Rule of thumb:** if it doesn't fit comfortably on one line, write the loop.

---
## 9. Practice

**1.** From `range(1, 21)`, build a list of the numbers divisible by 3.

In [ ]:
# Exercise 1

**2.** Given `words = ["python", "is", "fun", "and", "powerful"]`, build a list of the words longer than 3 letters, in upper case.

In [ ]:
# Exercise 2

**3.** Given `temps_c = {"mon": 30, "tue": 35, "wed": 28}`, build a dict of the same temperatures in Fahrenheit (`c * 9 / 5 + 32`).

In [ ]:
# Exercise 3

**4.** Given `marks = {"asha": 92, "ravi": 45, "monal": 78, "alex": 30}`, build a dict mapping each student to `"pass"` (50 or more) or `"fail"`.

In [ ]:
# Exercise 4

**5.** Read `practice_files/marks.txt` (created in notebook 03) and build `{name: mark}` with a dict comprehension, skipping the bad lines. Hint: a comprehension can't use `try` / `except`, so filter the lines with `if` first, for example by checking `line.count(",") == 1` and `.isdigit()`.

In [ ]:
# Exercise 5